# Python Developer Market Analysis
Exploratory Data Analysis of the Stack Overflow Developer Survey 2025

Author: Vitalii  
Dataset: Stack Overflow Developer Survey 2025  
Total respondents: 49,191

## Project Goal

The purpose of this analysis is to explore the global Python developer market using the Stack Overflow Developer Survey dataset.

Key objectives:

- Analyze Python popularity among developers
- Explore salary distribution and geographic differences
- Examine education background of high-earning developers
- Identify industries with high-income remote workers

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# import pandas as pd

# public = '../data/survey_results_public.csv'
# schema = '../data/survey_results_schema.csv'

# df_public = pd.read_csv(public, low_memory=False)
# df_schema = pd.read_csv(schema)
# df_public.shape

### Task 1. Total number of respondents

Each row in the dataset represents one survey respondent.  
The total number of respondents is calculated as the number of rows in the dataset.

In [ ]:
# 1. Loading the dataset and initial preview
# Using the survey_results_public.csv file which contains the survey responses
df_public = pd.read_csv(public, low_memory=False)
df_public.head()

,ResponseId,MainBranch,Age,EdLevel,Employment,EmploymentAddl,WorkExp,LearnCodeChoose,LearnCode,LearnCodeAI,...,AIAgentOrchestration,AIAgentOrchWrite,AIAgentObserveSecure,AIAgentObsWrite,AIAgentExternal,AIAgentExtWrite,AIHuman,AIOpen,ConvertedCompYearly,JobSat
0,1,I am a developer by profession,25-34 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Employed,"Caring for dependents (children, elderly, etc.)",8.0,"Yes, I am not new to coding but am learning ne...",Online Courses or Certification (includes all ...,"Yes, I learned how to use AI-enabled tools for...",...,Vertex AI,NaN,NaN,NaN,ChatGPT,NaN,When I don’t trust AI’s answers,"Troubleshooting, profiling, debugging",61256.0,10.0
1,2,I am a developer by profession,25-34 years old,"Associate degree (A.A., A.S., etc.)",Employed,NaN,2.0,"Yes, I am not new to coding but am learning ne...",Online Courses or Certification (includes all ...,"Yes, I learned how to use AI-enabled tools for...",...,NaN,NaN,NaN,NaN,NaN,NaN,When I don’t trust AI’s answers;When I want to...,All skills. AI is a flop.,104413.0,9.0
2,3,I am a developer by profession,35-44 years old,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)","Independent contractor, freelancer, or self-em...",None of the above,10.0,"Yes, I am not new to coding but am learning ne...",Online Courses or Certification (includes all ...,"Yes, I learned how to use AI-enabled tools for...",...,NaN,NaN,NaN,NaN,ChatGPT;Claude Code;GitHub Copilot;Google Gemini,NaN,When I don’t trust AI’s answers;When I want to...,"Understand how things actually work, problem s...",53061.0,8.0
3,4,I am a developer by profession,35-44 years old,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Employed,None of the above,4.0,"Yes, I am not new to coding but am learning ne...","Other online resources (e.g. standard search, ...","Yes, I learned how to use AI-enabled tools for...",...,NaN,NaN,NaN,NaN,ChatGPT;Claude Code,NaN,When I don’t trust AI’s answers;When I want to...,NaN,36197.0,6.0
4,5,I am a developer by profession,35-44 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)","Independent contractor, freelancer, or self-em...","Caring for dependents (children, elderly, etc.)",21.0,"No, I am not new to coding and did not learn n...",NaN,"Yes, I learned how to use AI-enabled tools for...",...,NaN,NaN,NaN,NaN,NaN,NaN,When I don’t trust AI’s answers,"critical thinking, the skill to define the tas...",60000.0,7.0


In [ ]:
# 2. Calculating the total number of respondents
# Using the .shape attribute where the first element [0] represents the number of rows
total_respondents = df_public.shape[0]
# 3. Displaying the result
print(f"Total number of respondents in Stack Overflow Developer Survey 2025: {total_respondents}")

Total number of respondents in Stack Overflow Developer Survey 2025: 49191


I observed a difference of 182 rows from the official summary (49090).
This indicates that the official results likely underwent further cleaning of outliers that were present in the raw data.

### Task 2. Completeness of survey responses

This task identifies how many respondents answered **all questions** in the survey.

The list of survey questions is obtained from the `survey_results_schema` file (`qname` field).  
Only questions that are present in both the schema and the main dataset are considered by using **set intersection**.

A respondent is counted as having complete responses if none of the selected survey questions contain missing values.


In [ ]:
# 1. Loading the survey schema
df_schema = pd.read_csv(schema)
df_schema.head()

,qid,qname,question,type,sub,sq_id
0,QID18,TechEndorse_1,What attracts you to a technology or causes yo...,RO,AI integration or AI Agent capabilities,1.0
1,QID18,TechEndorse_2,What attracts you to a technology or causes yo...,RO,Easy-to-use API,2.0
2,QID18,TechEndorse_3,What attracts you to a technology or causes yo...,RO,Robust and complete API,3.0
3,QID18,TechEndorse_4,What attracts you to a technology or causes yo...,RO,Customizable and manageable codebase,4.0
4,QID18,TechEndorse_5,What attracts you to a technology or causes yo...,RO,Reputation for quality,5.0


In [ ]:
# 2. Getting the list of questions from the 'qname' field
# Converting to sets for efficient intersection as per project requirements
questions_set = set(df_schema['qname'])
columns_set = set(df_public.columns)

# 3. Applying Intersection to find only valid survey questions present in the dataset
survey_questions = list(questions_set & columns_set)

# 4. Calculating the number of respondents with complete answers (no NaNs)
# We use .notna().all(axis=1) to verify completeness across all selected columns
mask_complete = df_public[survey_questions].notna().all(axis=1)
complete_count = int(mask_complete.sum())

# 5. Displaying the results
print(f"Total valid survey questions identified: {len(survey_questions)}")
print(f"Number of respondents who answered every single question: {complete_count}")

#The Stack Overflow survey includes conditional questions, meaning not every respondent is shown every question.
#Therefore, it is expected that the number of respondents with non-missing values for all survey questions can be very low or even 0.

Total valid survey questions identified: 126
Number of respondents who answered every single question: 0


### Task 3. Statistical analysis of respondents’ work experience

This task analyzes respondents’ work experience using descriptive statistics.

The following measures of central tendency are calculated for the `WorkExp` field:
- Mean
- Median
- Mode

Missing or non-numeric values are handled appropriately to ensure accurate statistical calculations.


In [ ]:
# 1. Calculating missing values metrics
missing_count = df_public['WorkExp'].isna().sum()
missing_percentage = (missing_count / len(df_public)) * 100

# 2. Calculating Central Tendency metrics (Mean, Median, Mode)
# Mastery: .mean() and .median() skip NaNs by default.
# For .mode(), we take the first value [0] as it returns a Series.
work_exp_mean = df_public['WorkExp'].mean()
work_exp_median = df_public['WorkExp'].median()
work_exp_mode = df_public['WorkExp'].mode()[0]

# 3. Displaying results with English labels
print(f"Missing values handled: {missing_count} ({missing_percentage:.2f}%)")
print(f"Mean (Average) Experience: {work_exp_mean:.2f} years")
print(f"Median Experience: {work_exp_median} years")
print(f"Mode (Most common) Experience: {work_exp_mode} years")

Missing values handled: 6298 (12.80%)
Mean (Average) Experience: 13.37 years
Median Experience: 10.0 years
Mode (Most common) Experience: 10.0 years


### Task 4. Analysis of remote work

This task determines the number of respondents who work remotely.

The dataset is filtered based on the column describing work format.  
Respondents who selected a remote work option are identified, and their total count is calculated.


In [ ]:
# Step 1: Identify the column and explore unique values
# We look for 'RemoteWork' and check what people actually answered
df_public['RemoteWork'].unique()

array(['Remote', 'Hybrid (some in-person, leans heavy to flexibility)',
       nan, 'In-person', 'Hybrid (some remote, leans heavy to in-person)',
       'Your choice (very flexible, you can come in when you want or just as needed)'],
      dtype=object)

In [ ]:
# 1. Creating a boolean series (True for 'Remote', False otherwise)
# This is our Boolean Mask
mask_remote = df_public['RemoteWork'] == 'Remote'

# 2. Counting True values using .sum()
remote_count = mask_remote.sum()

# 3. Final output for the cell
print(f"Total remote workers: {remote_count}")

Total remote workers: 10931


### Task 5. Popularity of Python among respondents

This task calculates the percentage of respondents who use **Python** as a programming language.

Since the programming languages field may contain multiple values per respondent,  
the analysis checks whether Python appears in the list of languages for each respondent.

The final result is presented as a percentage.


In [ ]:
# 1. Searching for columns that mention 'language' in their description
# case=False: ensures we don't miss any entries due to capitalization
# na=False: treats missing survey responses as False
df_schema[df_schema['question'].str.contains('language', case=False, na=False)]

,qid,qname,question,type,sub,sq_id
55,QID10,LearnCodeChoose,Did you begin learning to code or learn a new ...,MC,NaN,NaN
75,QID44,LanguageChoice,"Did you work with programming, scripting and m...",MC,NaN,NaN
76,QID34,Language,"Which <b>programming, scripting, and markup la...",Matrix,NaN,NaN
77,QID51,LanguagesHaveEntry,Was the programming language you use not liste...,TE,NaN,NaN
78,QID52,LanguagesWantEntry,Was the programming language you want to use n...,TE,NaN,NaN
105,QID50,AIModelsChoice,Did you utilize or integrate specific Large La...,MC,NaN,NaN


In [ ]:
# 2. Getting all column names that contain 'Lang'
candidate_cols = [col for col in df_public.columns if 'Lang' in col]
print(candidate_cols)

['LanguageChoice', 'LanguageHaveWorkedWith', 'LanguageWantToWorkWith', 'LanguageAdmired', 'LanguagesHaveEntry', 'LanguagesWantEntry']


In [ ]:
# 3.Exploratory Analysis: Verifying which column contains more comprehensive data
# Testing 'LanguagesHaveEntry' vs 'LanguagesHaveWorkedWith'
count_entry = df_public['LanguagesHaveEntry'].count()
count_worked_with = df_public['LanguageHaveWorkedWith'].count()

print(f"Non-null records in 'LanguagesHaveEntry': {count_entry}")
print(f"Non-null records in 'LanguageHaveWorkedWith': {count_worked_with}")

Non-null records in 'LanguagesHaveEntry': 3982
Non-null records in 'LanguageHaveWorkedWith': 31671


The column LanguageHaveWorkedWith contains significantly more entries, making it a more reliable source for analyzing the programming languages used by developers. Therefore, this column will be used for all subsequent cleaning and filtering steps.

In [ ]:
# 3. Identify users who have Python in 'LanguagesHaveWorkedWith'
# case=False: ensures we don't miss any entries due to capitalization
# na=False: treats missing survey responses as False
mask_worked = df_public['LanguageHaveWorkedWith'].str.contains('Pyth', case=False, na=False)

print(f"Users with Python in 'LanguageHaveWorkedWith': {mask_worked.sum()}")

Users with Python in 'LanguageHaveWorkedWith': 18466


In [ ]:
# 3. Identifying users who have 'Python' in their language stack
# case=False: ensures we don't miss any entries due to capitalization
# na=False: treats missing survey responses as False
python_mask = df_public['LanguageHaveWorkedWith'].str.contains('Pyth', case=False, na=False)
python_users_count = python_mask.sum()

# 4. Total number of respondents who provided information about their languages
total_responses = df_public['LanguageHaveWorkedWith'].count()

# 5. Calculating the percentage
python_percentage = (python_users_count / total_responses) * 100

# 6. Final output in the required format
print(f"Total valid responses: {total_responses}")
print(f"Users who program in Python: {python_users_count}")
print(f"Percentage of Python users: {python_percentage:.1f}%")

Total valid responses: 31671
Users who program in Python: 18466
Percentage of Python users: 58.3%


### Task 6. Analysis of learning paths (online courses)

This task identifies how many respondents learned programming through **online courses**.

The analysis accounts for the possibility of multiple learning methods per respondent  
and counts all respondents who selected online courses as one of their learning paths.


In [ ]:
# 1.Searching for the education/learning column
# Filtering columns that contain 'Learn' to find the right one
learn_cols = [col for col in df_public.columns if 'Learn' in col]
print(learn_cols)

['LearnCodeChoose', 'LearnCode', 'LearnCodeAI', 'AILearnHow']


In [ ]:
# 2. Checking unique values (briefly) to find the exact string for online courses
print(df_public['LearnCode'].unique()[:5])

['Online Courses or Certification (includes all media types);Other online resources (e.g. standard search, forum, online community)'
 'Online Courses or Certification (includes all media types);Other online resources (e.g. standard search, forum, online community);Books / Physical media;Videos (not associated with specific online course or certification);Stack Overflow or Stack Exchange'
 'Online Courses or Certification (includes all media types);Videos (not associated with specific online course or certification);Technical documentation (is generated for/by the tool or system)'
 'Other online resources (e.g. standard search, forum, online community);Videos (not associated with specific online course or certification);Stack Overflow or Stack Exchange;Technical documentation (is generated for/by the tool or system);AI CodeGen tools or AI-enabled apps'
 nan]


In [ ]:
# 3. Define target column and the specific category
target_col = 'LearnCode'
precise_term = 'Online Courses'

# 4. Create a boolean mask to handle multiple selections
online_courses_mask = df_public[target_col].str.contains(precise_term, case=False, na=False)

# 5. Calculate the count of specific respondents
online_courses_count = online_courses_mask.sum()

# 6. Calculate the total number of respondents who answered this question (non-null)
total_responses = df_public[target_col].count()

# 7. Calculate the percentage
online_percentage = (online_courses_count / total_responses) * 100

# 8. Display the results
print(f"Total valid responses for '{target_col}': {total_responses}")
print(f"Total respondents who used Online Courses or Certification: {online_courses_count}")
print(f"Percentage of respondents: {online_percentage:.1f}%")

Total valid responses for 'LearnCode': 33556
Total respondents who used Online Courses or Certification: 10973
Percentage of respondents: 32.7%


### Task 7. Compensation analysis of Python developers by country

This task analyzes compensation levels of respondents who use Python, grouped by country.

For each country, the following metrics are calculated:
- Average annual compensation
- Median annual compensation

Only respondents with valid compensation data are included in the analysis.


In [ ]:
# 1. Applying the mask to filter Python developers and grouping by Country
# 2. Calculating mean and median for 'ConvertedCompYearly'
# 3. Removing countries with missing compensation data (NaN)
# 4. Rounding values to 2 decimal places for better readability
# 5. Sorting by median compensation in descending order
# 6. Moving 'Country' from index to a regular column using .reset_index()

compensation_table = (df_public[mask_worked]
                            .groupby('Country')['ConvertedCompYearly']
                            .agg(['mean', 'median','count'])
                            .dropna()
                            .round(2)
                            .sort_values(by='median', ascending=False)
                            .reset_index())

# 2. Renaming columns for a professional report
compensation_table.columns = ['Country', 'Average Comp', 'Median Comp', 'Number of Respondents']

# 3.Calculating the total sample size for Python compensation analysis
total_responses = df_public[mask_worked]['ConvertedCompYearly'].dropna().count()
print(f"Total valid Python compensation responses: {total_responses:,}")
print("-" * 50)

# 3. Displaying the top of the table
print("Top Countries by Python Compensation (with respondent count):")
display(compensation_table.head(10))

Total valid Python compensation responses: 12,765
--------------------------------------------------
Top Countries by Python Compensation (with respondent count):


,Country,Average Comp,Median Comp,Number of Respondents
0,Oman,390135.00,390135.0,2
1,Andorra,226103.50,226103.5,2
2,United States of America,173298.59,150000.0,3126
3,Israel,135828.37,142594.0,104
4,Switzerland,156456.60,142592.0,240
5,Nomadic,120131.57,139218.0,7
6,Ireland,120523.92,116015.0,74
7,Luxembourg,116014.71,109054.0,7
8,Kyrgyzstan,106008.50,106008.5,2
9,Belize,102121.00,102121.0,1


Observations on Data Reliability:

The table above ranks countries by their median compensation for Python developers. However, it is important to note that the top positions (e.g., Oman, Andorra, and Belize) are occupied by countries with a very small sample size (1-2 respondents). These figures may represent individual outliers rather than the broader market average for those regions.

In contrast, countries with a significant number of respondents, such as the United States (3,126), Israel (104), and Switzerland (240), provide much more statistically reliable data for salary benchmarking.

Note: I have chosen to present the raw data for transparency. If a more stable market comparison is required, a filter (e.g., count > 30) could be applied to exclude small-sample outliers.

### Task 8. Education levels of top-paid respondents

This task examines the education levels of the **top 5 respondents with the highest compensation**.

Respondents are sorted by annual compensation in descending order,  
and the education levels of the top earners are extracted.


In [ ]:
# We are using raw compensation data without upper-bound filtering
# to observe the maximum values reported in the dataset.

# 1. Filter: Ensure respondents listed at least one programming language
# This ensures we are looking at technical profiles only.
mask_has_languages = df_public['LanguageHaveWorkedWith'].notna()

# 2. Sort by compensation descending and select top 5
top_5_earners = (df_public[mask_has_languages]
                 .sort_values(by='ConvertedCompYearly', ascending=False)
                 .head(5)[['ConvertedCompYearly', 'EdLevel']])

# 3. Reset index for ranking
top_5_earners.index = range(1, 6)

# 4. Calculate the sum of valid responses
total_language_responses = mask_has_languages.sum()

# 3. Display the result with a clear label
print(f"Total respondents with valid language data: {total_language_responses:,}")
print("-" * 50)

print("Top 5 Earners - technical profiles only:")
top_5_earners

Total respondents with valid language data: 31,671
--------------------------------------------------
Top 5 Earners - technical profiles only:


,ConvertedCompYearly,EdLevel
1,33552715.0,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)"
2,13921760.0,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)"
3,10000000.0,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)"
4,9531653.0,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)"
5,6890299.0,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)"


# Analysis of the $33.5M Entry (Raw Data Context)

Observation: The highest compensation entry is $33,552,715, which corresponds to approximately 1,400,000,000 UAH (Ukrainian Hryvnia).

Critical Insight: While the currency conversion itself might be technically accurate, the source value in UAH is fundamentally unrealistic for a personal annual salary.

•	The Scale: A salary of 1.4 billion UAH per year would mean a monthly income of approximately 116 million UAH.

•	Conclusion: It is highly probable that this respondent entered a placeholder value, a joke, or perhaps the total valuation/revenue of a company rather than their personal compensation.

### Task 9. Python popularity by age group (optional)

This task analyzes Python usage across different age groups.

For each age category, the percentage of respondents who reported using Python is calculated.  
This helps identify which age groups show the highest adoption of Python.


In [ ]:
# 1. Calculate the actual number of valid responses (N-size)
# We count only those who provided both Age and Language data
total_valid_responses = df_public.dropna(subset=['Age', 'LanguageHaveWorkedWith']).shape[0]

# 2. Group the 'mask_worked' series by the 'Age' column
python_popularity = (mask_worked.groupby(df_public['Age'])
                     .mean() * 100).round(2).reset_index()

# 3. Rename columns for professional reporting
python_popularity.columns = ['Age Category', 'Python Popularity (%)']

# 4. Sort values to visualize trends (from highest popularity to lowest)
python_popularity = python_popularity.sort_values(by='Python Popularity (%)', ascending=False)

# 5. Display the result
print(f"Based on {total_valid_responses:,} valid responses.")
print("-" * 50)
print(f"Final Analysis: Python Popularity Across Age Groups")
python_popularity

Based on 31,671 valid responses.
--------------------------------------------------
Final Analysis: Python Popularity Across Age Groups


,Age Category,Python Popularity (%)
0,18-24 years old,40.00
3,45-54 years old,38.63
4,55-64 years old,37.24
1,25-34 years old,36.94
2,35-44 years old,36.72
5,65 years or older,31.63
6,Prefer not to say,31.22


### Task 10. Industry analysis of high-paid remote workers (optional)

This task focuses on respondents who:
- Earn more than the 75th percentile of annual compensation
- Work remotely

For this group, the most common industries are identified and ranked by frequency.


In [ ]:
# [Step 10.1]: Comparative Salary Compensation Analysis
# We compare the 75th percentile before and after cleaning the data

# 1. Raw compensation (all respondents)
raw_comp = df_public['ConvertedCompYearly'].quantile(0.75)

# 2. Cleaning: removing respondents who didn't specify their languages
df_clean_langs = df_public.dropna(subset=['LanguageHaveWorkedWith']).copy()

# 3. Cleaned threshold (only respondents with listed languages)
cleaned_comp =df_clean_langs['ConvertedCompYearly'].quantile(0.75)

# Displaying the results for comparison
print(f"75th Percentile (Raw Data): ${raw_comp:,.2f}")
print(f"75th Percentile (Cleaned Data): ${cleaned_comp:,.2f}")
print(f"Difference: ${cleaned_comp - raw_comp:,.2f}")

75th Percentile (Raw Data): $120,596.00
75th Percentile (Cleaned Data): $122,527.00
Difference: $1,931.00


The 75th percentile threshold increased from `$120,596` to `$122,527` after removing respondents who did not specify their programming languages. This indicates a positive correlation between data completeness and compensation levels. By using the higher threshold from the cleaned dataset (df_clean_langs), we ensure our analysis focuses on a more verified and professional segment of the high-earning workforce.

In [ ]:
# [Step 10.2]:
# 1. Total valid responses for this specific analysis
total_valid_responses = df_public.dropna(subset=['LanguageHaveWorkedWith', 'ConvertedCompYearly', 'Industry', 'RemoteWork']).shape[0]

# 2. Compensation calculation
cleaned_comp = df_clean_langs['ConvertedCompYearly'].quantile(0.75)

# 3. Filtering
mask_high_salary = df_clean_langs['ConvertedCompYearly'] > cleaned_comp
mask_remote = df_clean_langs['RemoteWork'] == 'Remote'
final_filter = mask_high_salary & mask_remote

# 4. Industry distribution
industry_analysis = (
    df_clean_langs[final_filter]['Industry']
    .value_counts()
    .reset_index()
)
industry_analysis.columns = ['Industry', 'Frequency']

# Output
print(f"Total valid responses analyzed: {total_valid_responses:,}")
print("-" * 60)
print(f"75th Percentile Salary Compensation: ${cleaned_comp:,.2f}")
display(industry_analysis)

Total valid responses analyzed: 19,543
------------------------------------------------------------
75th Percentile Salary Compensation: $122,527.00


,Industry,Frequency
0,Software Development,1089
1,Fintech,176
2,Healthcare,172
3,Other:,167
4,"Internet, Telecomm or Information Services",130
5,Banking/Financial Services,79
6,Media & Advertising Services,72
7,Government,70
8,"Transportation, or Supply Chain",59
9,Retail and Consumer Services,57
